# 02. Model Implementation - GAE & VGAE

**Đồ án:** GNN Protein Function Prediction  
**Môn học:** IS353 - Mạng Xã Hội

## Mục tiêu
1. Implement R-GCN Encoder (2 layers)
2. Implement DistMult Decoder
3. Implement GAE (Graph Autoencoder)
4. Implement VGAE (Variational GAE)
5. Negative Sampling

## 1. Setup & Installation

In [ ]:
# Install dependencies (chạy trên Colab)
!pip install torch torch-geometric -q
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.0.0+cu118.html -q
!pip install pandas numpy matplotlib seaborn -q

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import RGCNConv
from torch_geometric.data import Data
import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Load and Preprocess Data

In [ ]:
# Run notebook 01 first to generate this file
# Or download directly
import os
import urllib.request
import gzip

os.makedirs('data', exist_ok=True)

if not os.path.exists('data/polypharmacy.csv'):
    url = 'http://snap.stanford.edu/biodata/datasets/10017/files/ChChSe-Decagon_polypharmacy.csv.gz'
    gz_path = 'data/polypharmacy.csv.gz'
    print("Downloading dataset...")
    urllib.request.urlretrieve(url, gz_path)
    with gzip.open(gz_path, 'rb') as f_in:
        with open('data/polypharmacy.csv', 'wb') as f_out:
            f_out.write(f_in.read())
    os.remove(gz_path)
    print("Done!")

# Load data
df = pd.read_csv('data/polypharmacy.csv')
df.columns = ['Drug1', 'Drug2', 'SideEffect']
print(f"Loaded {len(df):,} interactions")

In [ ]:
# Use top N relations for faster training
TOP_N_RELATIONS = 50  # Adjust based on GPU memory

relation_counts = df['SideEffect'].value_counts()
top_relations = relation_counts.head(TOP_N_RELATIONS).index.tolist()
df_filtered = df[df['SideEffect'].isin(top_relations)].copy()

print(f"Using top {TOP_N_RELATIONS} relations")
print(f"Filtered edges: {len(df_filtered):,}")

In [ ]:
# Create node and relation mappings
all_drugs = sorted(set(df_filtered['Drug1']) | set(df_filtered['Drug2']))
all_relations = sorted(set(df_filtered['SideEffect']))

drug_to_idx = {drug: idx for idx, drug in enumerate(all_drugs)}
relation_to_idx = {rel: idx for idx, rel in enumerate(all_relations)}
idx_to_drug = {idx: drug for drug, idx in drug_to_idx.items()}
idx_to_relation = {idx: rel for rel, idx in relation_to_idx.items()}

num_nodes = len(drug_to_idx)
num_relations = len(relation_to_idx)

print(f"Nodes: {num_nodes}")
print(f"Relations: {num_relations}")

In [ ]:
# Build edge lists
edges = []
for _, row in df_filtered.iterrows():
    src = drug_to_idx[row['Drug1']]
    dst = drug_to_idx[row['Drug2']]
    rel = relation_to_idx[row['SideEffect']]
    edges.append((src, dst, rel))
    edges.append((dst, src, rel))  # Undirected

edges = list(set(edges))  # Remove duplicates
print(f"Total edges (undirected): {len(edges):,}")

In [ ]:
# Train/Val/Test split (80/10/10)
np.random.seed(42)
np.random.shuffle(edges)

n = len(edges)
train_edges = edges[:int(0.8*n)]
val_edges = edges[int(0.8*n):int(0.9*n)]
test_edges = edges[int(0.9*n):]

print(f"Train: {len(train_edges):,}")
print(f"Val: {len(val_edges):,}")
print(f"Test: {len(test_edges):,}")

In [ ]:
# Convert to tensors
def edges_to_tensors(edge_list):
    src = torch.tensor([e[0] for e in edge_list], dtype=torch.long)
    dst = torch.tensor([e[1] for e in edge_list], dtype=torch.long)
    rel = torch.tensor([e[2] for e in edge_list], dtype=torch.long)
    edge_index = torch.stack([src, dst], dim=0)
    return edge_index, rel

train_edge_index, train_edge_type = edges_to_tensors(train_edges)
val_edge_index, val_edge_type = edges_to_tensors(val_edges)
test_edge_index, test_edge_type = edges_to_tensors(test_edges)

# Node features (use identity or learnable embeddings)
x = torch.eye(num_nodes)  # One-hot (simple)

print(f"Edge index shape: {train_edge_index.shape}")
print(f"Edge type shape: {train_edge_type.shape}")

## 3. R-GCN Encoder

**R-GCN (Relational Graph Convolutional Network)**
- Extends GCN to handle multiple edge types
- Each relation has its own weight matrix

Formula:
$$h_i^{(l+1)} = \sigma \left( \sum_{r \in R} \sum_{j \in N_i^r} \frac{1}{c_{i,r}} W_r^{(l)} h_j^{(l)} + W_0^{(l)} h_i^{(l)} \right)$$

In [ ]:
class RGCNEncoder(nn.Module):
    """
    R-GCN Encoder with 2 layers
    
    For VGAE: outputs both mean (mu) and log variance (logvar)
    """
    def __init__(self, in_channels, hidden_channels, out_channels, num_relations, 
                 num_bases=None, dropout=0.0, variational=False):
        super().__init__()
        
        self.variational = variational
        self.dropout = dropout
        
        # Layer 1: input -> hidden
        self.conv1 = RGCNConv(
            in_channels, hidden_channels, 
            num_relations=num_relations,
            num_bases=num_bases
        )
        
        # Layer 2: hidden -> output (mean)
        self.conv2_mu = RGCNConv(
            hidden_channels, out_channels,
            num_relations=num_relations,
            num_bases=num_bases
        )
        
        # For VGAE: additional layer for log variance
        if variational:
            self.conv2_logvar = RGCNConv(
                hidden_channels, out_channels,
                num_relations=num_relations,
                num_bases=num_bases
            )
    
    def forward(self, x, edge_index, edge_type):
        # Layer 1
        x = self.conv1(x, edge_index, edge_type)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        
        # Layer 2
        mu = self.conv2_mu(x, edge_index, edge_type)
        
        if self.variational:
            logvar = self.conv2_logvar(x, edge_index, edge_type)
            return mu, logvar
        else:
            return mu

print("✅ RGCNEncoder defined")

## 4. DistMult Decoder

**DistMult** scoring function for multi-relational link prediction:

$$score(u, r, v) = h_u^T \cdot diag(R_r) \cdot h_v = \sum_i h_u^{(i)} \cdot R_r^{(i)} \cdot h_v^{(i)}$$

Where:
- $h_u, h_v$: node embeddings
- $R_r$: diagonal relation matrix (learnable)

In [ ]:
class DistMultDecoder(nn.Module):
    """
    DistMult Decoder for multi-relational link prediction
    
    score(u, r, v) = h_u ⊙ R_r ⊙ h_v (element-wise product, then sum)
    """
    def __init__(self, num_relations, embedding_dim):
        super().__init__()
        
        # Relation embeddings (diagonal matrices)
        self.relation_embeddings = nn.Parameter(
            torch.Tensor(num_relations, embedding_dim)
        )
        nn.init.xavier_uniform_(self.relation_embeddings)
    
    def forward(self, z, edge_index, edge_type):
        """
        Args:
            z: Node embeddings [num_nodes, embedding_dim]
            edge_index: [2, num_edges]
            edge_type: [num_edges]
        
        Returns:
            scores: [num_edges]
        """
        head = z[edge_index[0]]  # [num_edges, dim]
        tail = z[edge_index[1]]  # [num_edges, dim]
        rel = self.relation_embeddings[edge_type]  # [num_edges, dim]
        
        # DistMult: element-wise product, then sum
        scores = (head * rel * tail).sum(dim=1)
        return scores
    
    def forward_all(self, z, edge_type_idx):
        """
        Compute scores for ALL possible node pairs for a specific relation
        (used for evaluation)
        """
        rel = self.relation_embeddings[edge_type_idx]  # [dim]
        # z: [num_nodes, dim]
        # scores[i,j] = (z[i] * rel * z[j]).sum()
        return torch.mm(z * rel, z.t())

print("✅ DistMultDecoder defined")

## 5. GAE (Graph Autoencoder)

**GAE** combines:
- Encoder: R-GCN (deterministic)
- Decoder: DistMult

Loss: Binary Cross Entropy between predicted and actual edges

In [ ]:
class GAE(nn.Module):
    """
    Graph Autoencoder
    
    Encoder: R-GCN
    Decoder: DistMult
    """
    def __init__(self, in_channels, hidden_channels, out_channels, 
                 num_relations, num_bases=None, dropout=0.0):
        super().__init__()
        
        self.encoder = RGCNEncoder(
            in_channels, hidden_channels, out_channels,
            num_relations, num_bases, dropout, variational=False
        )
        self.decoder = DistMultDecoder(num_relations, out_channels)
    
    def encode(self, x, edge_index, edge_type):
        return self.encoder(x, edge_index, edge_type)
    
    def decode(self, z, edge_index, edge_type):
        return self.decoder(z, edge_index, edge_type)
    
    def forward(self, x, edge_index, edge_type):
        z = self.encode(x, edge_index, edge_type)
        return z

print("✅ GAE defined")

## 6. VGAE (Variational Graph Autoencoder)

**VGAE** adds variational inference:
- Encoder outputs $\mu$ and $\log \sigma^2$
- Reparameterization trick: $z = \mu + \sigma \cdot \epsilon$, where $\epsilon \sim N(0, 1)$

Loss: Reconstruction Loss + KL Divergence

$$\mathcal{L} = -\mathbb{E}_{q(Z|X,A)}[\log p(A|Z)] + KL[q(Z|X,A) \| p(Z)]$$

In [ ]:
class VGAE(nn.Module):
    """
    Variational Graph Autoencoder
    
    Encoder: R-GCN (outputs mu and logvar)
    Decoder: DistMult
    """
    def __init__(self, in_channels, hidden_channels, out_channels,
                 num_relations, num_bases=None, dropout=0.0):
        super().__init__()
        
        self.encoder = RGCNEncoder(
            in_channels, hidden_channels, out_channels,
            num_relations, num_bases, dropout, variational=True
        )
        self.decoder = DistMultDecoder(num_relations, out_channels)
    
    def reparameterize(self, mu, logvar):
        """
        Reparameterization trick:
        z = mu + std * epsilon
        """
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        else:
            return mu
    
    def encode(self, x, edge_index, edge_type):
        mu, logvar = self.encoder(x, edge_index, edge_type)
        z = self.reparameterize(mu, logvar)
        return z, mu, logvar
    
    def decode(self, z, edge_index, edge_type):
        return self.decoder(z, edge_index, edge_type)
    
    def kl_loss(self, mu, logvar):
        """
        KL Divergence: KL[q(z|x) || p(z)]
        where q(z|x) = N(mu, sigma^2) and p(z) = N(0, 1)
        """
        return -0.5 * torch.mean(
            torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)
        )
    
    def forward(self, x, edge_index, edge_type):
        z, mu, logvar = self.encode(x, edge_index, edge_type)
        return z, mu, logvar

print("✅ VGAE defined")

## 7. Negative Sampling

For each positive edge (u, r, v), we create negative edges by:
- Keeping u and r fixed, replacing v with random node v'

In [ ]:
def negative_sampling(edge_index, edge_type, num_nodes, num_neg_samples=1):
    """
    Generate negative samples by corrupting tail nodes
    
    Args:
        edge_index: [2, num_edges]
        edge_type: [num_edges]
        num_nodes: total number of nodes
        num_neg_samples: number of negative samples per positive edge
    
    Returns:
        neg_edge_index: [2, num_edges * num_neg_samples]
        neg_edge_type: [num_edges * num_neg_samples]
    """
    num_edges = edge_index.size(1)
    
    # Keep head and relation, corrupt tail
    neg_heads = edge_index[0].repeat(num_neg_samples)
    neg_types = edge_type.repeat(num_neg_samples)
    neg_tails = torch.randint(0, num_nodes, (num_edges * num_neg_samples,), device=edge_index.device)
    
    neg_edge_index = torch.stack([neg_heads, neg_tails], dim=0)
    
    return neg_edge_index, neg_types

# Test
neg_idx, neg_type = negative_sampling(train_edge_index, train_edge_type, num_nodes, num_neg_samples=1)
print(f"Positive edges: {train_edge_index.shape[1]:,}")
print(f"Negative edges: {neg_idx.shape[1]:,}")

## 8. Model Testing

In [ ]:
# Hyperparameters
HIDDEN_DIM = 64
EMBEDDING_DIM = 32
NUM_BASES = 30  # For basis decomposition (reduces parameters)
DROPOUT = 0.3

# Initialize models
gae_model = GAE(
    in_channels=num_nodes,
    hidden_channels=HIDDEN_DIM,
    out_channels=EMBEDDING_DIM,
    num_relations=num_relations,
    num_bases=NUM_BASES,
    dropout=DROPOUT
).to(device)

vgae_model = VGAE(
    in_channels=num_nodes,
    hidden_channels=HIDDEN_DIM,
    out_channels=EMBEDDING_DIM,
    num_relations=num_relations,
    num_bases=NUM_BASES,
    dropout=DROPOUT
).to(device)

print("GAE Model:")
print(gae_model)
print(f"\nTotal parameters: {sum(p.numel() for p in gae_model.parameters()):,}")

In [ ]:
print("\nVGAE Model:")
print(vgae_model)
print(f"\nTotal parameters: {sum(p.numel() for p in vgae_model.parameters()):,}")

In [ ]:
# Test forward pass
x_test = x.to(device)
edge_index_test = train_edge_index[:, :1000].to(device)
edge_type_test = train_edge_type[:1000].to(device)

# GAE
gae_model.eval()
with torch.no_grad():
    z_gae = gae_model.encode(x_test, edge_index_test, edge_type_test)
    scores_gae = gae_model.decode(z_gae, edge_index_test, edge_type_test)

print(f"GAE embedding shape: {z_gae.shape}")
print(f"GAE scores shape: {scores_gae.shape}")

# VGAE
vgae_model.eval()
with torch.no_grad():
    z_vgae, mu, logvar = vgae_model.encode(x_test, edge_index_test, edge_type_test)
    scores_vgae = vgae_model.decode(z_vgae, edge_index_test, edge_type_test)

print(f"\nVGAE embedding shape: {z_vgae.shape}")
print(f"VGAE mu shape: {mu.shape}")
print(f"VGAE logvar shape: {logvar.shape}")
print(f"VGAE scores shape: {scores_vgae.shape}")

## 9. Save Data for Next Notebooks

In [ ]:
# Save preprocessed data
import pickle

data_dict = {
    'num_nodes': num_nodes,
    'num_relations': num_relations,
    'drug_to_idx': drug_to_idx,
    'relation_to_idx': relation_to_idx,
    'idx_to_drug': idx_to_drug,
    'idx_to_relation': idx_to_relation,
    'train_edges': train_edges,
    'val_edges': val_edges,
    'test_edges': test_edges,
    'x': x,
    'train_edge_index': train_edge_index,
    'train_edge_type': train_edge_type,
    'val_edge_index': val_edge_index,
    'val_edge_type': val_edge_type,
    'test_edge_index': test_edge_index,
    'test_edge_type': test_edge_type,
}

with open('data/processed_data.pkl', 'wb') as f:
    pickle.dump(data_dict, f)

print("✅ Saved: data/processed_data.pkl")

---

## ✅ Checklist Phase 2

- [x] R-GCN Encoder (2 layers)
- [x] DistMult Decoder
- [x] GAE model
- [x] VGAE model (+ reparameterization)
- [x] Negative Sampling
- [x] Forward pass testing

**Next:** Phase 3 - Training + Grid Search + GAE vs VGAE Comparison